In [ ]:
# ------------------------------------------------------------------------------
# Cell 1: Install Dependencies
# ------------------------------------------------------------------------------
# NOTE: these are already present in kaggle
#!pip install fastapi uvicorn nest_asyncio requests huggingface_hub

In [ ]:
!pip install av comfyui-frontend-package comfyui-workflow-templates spandrel torchsde trampoline --no-deps

In [ ]:
# ------------------------------------------------------------------------------
# Cell 2: Imports & Configuration
# ------------------------------------------------------------------------------
import os
import sys
import json
import time
import uuid
import shutil
import threading
import subprocess
from pathlib import Path
from typing import Optional
import requests
from fastapi import FastAPI, Form, File, UploadFile, BackgroundTasks, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
import io
from PIL import Image
import nest_asyncio
import uvicorn
from huggingface_hub import snapshot_download

nest_asyncio.apply()

class CONFIG:
    # --- Your Original Config ---
    COMFY_DIR = "/kaggle/working/ComfyUI"
    COMFY_HOST = "localhost"
    COMFY_PORT = 8000
    PROXY_API_PREFIX = ""
    MANIFEST_PATH = "/kaggle/working/model_manifest.json"
    OUTPUTS_DIR = "/kaggle/working/comfy_outputs"
    AUTO_COPY_LOCAL_MODELS = True
    COMFY_REQUEST_TIMEOUT = 120
    IMAGE_EXTS = [".png", ".jpg", ".jpeg", ".webp"]

    # --- Infrastructure Additions ---
    ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
    ZROK_BINARY = "./zrok"
    ZROK_VERSION = "1.1.3"
    FACADE_PORT = 8001

Path(CONFIG.OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.COMFY_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ------------------------------------------------------------------------------
# Cell 3: Setup SDXL for ComfyUI (SINGLE CHECKPOINT FILE)
# ------------------------------------------------------------------------------
import os
import subprocess
from huggingface_hub import hf_hub_download

COMFY_DIR = "/kaggle/working/ComfyUI"

# 1. Clone ComfyUI
print("Cloning ComfyUI...")
subprocess.run(f"git clone https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}", shell=True, check=True)

# 2. Download SDXL Base 1.0 as SINGLE checkpoint file (ComfyUI format)
print("Downloading SDXL Base 1.0 checkpoint (6.9GB)...")
ckpt_dir = f"{COMFY_DIR}/models/checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)

# This downloads the merged single-file checkpoint
hf_hub_download(
    repo_id="stabilityai/stable-diffusion-xl-base-1.0",
    filename="sd_xl_base_1.0.safetensors",
    local_dir=ckpt_dir,
    local_dir_use_symlinks=False
)

print(f"✅ Checkpoint ready at {ckpt_dir}/sd_xl_base_1.0.safetensors")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 5: Infrastructure Helpers (Zrok & Comfy Process) - DIAGNOSTIC VERSION
# ------------------------------------------------------------------------------
def load_zrok_token():
    p = CONFIG.ZROK_TOKEN_PATH
    token = None
    if os.path.isfile(p):
        with open(p, "r", encoding="utf-8", errors="ignore") as f:
            token = f.read().strip()
    if not token:
        print(f"❌ zrok token not found at {p}. Tunneling will be skipped.")
    return token

def install_and_enable_zrok(token: Optional[str]):
    if not token: return
    print("Downloading zrok...")
    try:
        subprocess.run(f"wget -q https://github.com/openziti/zrok/releases/download/v{CONFIG.ZROK_VERSION}/zrok_{CONFIG.ZROK_VERSION}_linux_amd64.tar.gz", shell=True, check=True)
        subprocess.run("tar -xzf zrok_*_linux_amd64.tar.gz", shell=True, check=True)
        subprocess.run("chmod +x zrok", shell=True, check=True)
        subprocess.run("rm zrok_*_linux_amd64.tar.gz", shell=True, check=True)
        print("Enabling zrok...")
        res = subprocess.run(f'./zrok enable --headless "{token}"', shell=True, capture_output=True, text=True)
        if res.returncode != 0 and "already enabled" not in res.stderr:
            print("⚠️  zrok enable failed:", res.stderr)
        else:
            print("✅ zrok enabled")
    except Exception as e:
        print("⚠️  zrok setup failed:", e)

def start_zrok_share(port: int):
    print(f"Starting zrok share for port {port}...")
    p = subprocess.Popen([CONFIG.ZROK_BINARY, "share", "public", f"localhost:{port}", "--headless"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    time.sleep(5)
    status = subprocess.run([CONFIG.ZROK_BINARY, "overview"], capture_output=True, text=True)
    out = status.stdout + status.stderr
    import re
    m = re.search(r'(https?://[a-z0-9\-\.]+\.zrok\.io)', out)
    url = m.group(1) if m else None
    if url:
        print("✅ zrok tunnel:", url)
    else:
        print("⚠️ zrok URL not found in status.")
    return p, url

def start_comfyui_server():
    """Start ComfyUI with visible output for debugging"""
    print("🔧 Starting ComfyUI server...")
    print(f"   Command: python {CONFIG.COMFY_DIR}/main.py --listen 127.0.0.1 --port {CONFIG.COMFY_PORT}")
    
    cmd = [
        sys.executable, str(Path(CONFIG.COMFY_DIR)/"main.py"),
        "--listen", "127.0.0.1", 
        "--port", str(CONFIG.COMFY_PORT), 
        "--disable-auto-launch"
    ]
    
    # CRITICAL FIX: Let ComfyUI output print directly to console
    proc = subprocess.Popen(cmd, cwd=CONFIG.COMFY_DIR)
    
    # Wait and verify it actually started
    print("  Waiting for ComfyUI to respond...")
    max_wait = 30
    for i in range(max_wait):
        time.sleep(1)
        try:
            r = requests.get(f"http://{CONFIG.COMFY_HOST}:{CONFIG.COMFY_PORT}/system_stats", timeout=2)
            if r.status_code == 200:
                stats = r.json()
                print(f"✅ ComfyUI ALIVE after {i+1}s")
                print(f"   Stats: {stats}")
                return proc
        except Exception as e:
            if i % 5 == 0:
                print(f"  ...waiting ({i}s) - {type(e).__name__}")
    
    print("❌ ComfyUI did not respond within 30s")
    print("   Check the output above for errors")
    return proc

In [ ]:
# --------------------
# Cell 6 (replace entire cell) --------------------
# Application Logic: Comfy endpoints, manifest helpers, workflow builder,
# and submit_workflow_to_comfy (returns saved basenames inside CONFIG.OUTPUTS_DIR).
# --------------------
import json
import time
import requests
from pathlib import Path
from typing import Optional
from datetime import datetime

# Ensure CONFIG, probe_comfy, read_manifest, ensure_model_present_for_comfy,
# and build_minimal_t2i_workflow are available in the notebook environment.
# If CONFIG and helper functions are already defined in previous cells that's fine.
# We re-use those names here (CONFIG defined earlier in your notebook).

# ----------------------------
# Comfy HTTP helpers
# ----------------------------
def comfy_base_url():
    prefix = getattr(CONFIG, "PROXY_API_PREFIX", "") or ""
    return f"http://{CONFIG.COMFY_HOST}:{CONFIG.COMFY_PORT}{prefix}"

def comfy_post(path: str, json_payload=None, files=None, timeout=None):
    """POST helper for ComfyUI API"""
    url = comfy_base_url() + path
    timeout = timeout or CONFIG.COMFY_REQUEST_TIMEOUT
    r = requests.post(url, json=json_payload, files=files, timeout=timeout)
    r.raise_for_status()
    return r

def comfy_get(path: str, params=None, timeout=None):
    """GET helper for ComfyUI API"""
    url = comfy_base_url() + path
    timeout = timeout or CONFIG.COMFY_REQUEST_TIMEOUT
    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r

def probe_comfy():
    try:
        r = comfy_get("/system_stats", timeout=5)
        return r.json()
    except Exception as e:
        return {"error": str(e)}

# ----------------------------
# Manifest & model helpers
# ----------------------------
def read_manifest(path_or_dict):
    if isinstance(path_or_dict, dict):
        return path_or_dict
    p = Path(path_or_dict)
    if not p.exists():
        raise FileNotFoundError(f"manifest not found: {path_or_dict}")
    return json.loads(p.read_text())

def ensure_model_present_for_comfy(manifest):
    """
    Return the filename of the checkpoint (basename).
    Assumes you already downloaded the checkpoint into ComfyUI/models/checkpoints.
    """
    m = read_manifest(manifest)
    name = m.get("name", None)
    if not name:
        raise ValueError("manifest missing 'name' key")
    # Return basename only
    return Path(name).name

# ----------------------------
# Minimal T2I workflow builder for ComfyUI
# ----------------------------
def build_minimal_t2i_workflow(prompt: str, model_path_or_id: str,
                               height:int=768, width:int=768,
                               num_inference_steps:int=28, guidance_scale:float=7.5,
                               seed: Optional[int]=None):
    """
    Build a small ComfyUI workflow dict that:
      - loads checkpoint (CheckpointLoaderSimple)
      - encodes prompt with CLIPTextEncode
      - creates EmptyLatentImage
      - runs KSampler
      - decodes with VAEDecode
      - SaveImage to produce an output file
    Note: slot indices are used to wire nodes correctly.
    """
    import uuid
    if seed is None:
        # deterministic-ish unique seed
        seed = int(uuid.uuid4().int & (2**31-1))

    wf = {
        "1": {
            "class_type": "CheckpointLoaderSimple",
            "inputs": {
                "ckpt_name": model_path_or_id
            }
        },
        "2": {
            "class_type": "CLIPTextEncode",
            "inputs": {
                "text": prompt,
                "clip": ["1", 1]
            }
        },
        "3": {
            "class_type": "EmptyLatentImage",
            "inputs": {
                "width": width,
                "height": height,
                "batch_size": 1
            }
        },
        "4": {
            "class_type": "CLIPTextEncode",
            "inputs": {
                "text": "low quality, watermark, blurry",  # negative prompt
                "clip": ["1", 1]
            }
        },
        "5": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["1", 0],
                "positive": ["2", 0],
                "negative": ["4", 0],
                "latent_image": ["3", 0],
                "seed": seed,
                "steps": num_inference_steps,
                "cfg": guidance_scale,       # 'cfg' is the correct name
                "sampler_name": "euler",
                "scheduler": "normal",
                "denoise": 1.0
            }
        },
        "6": {
            "class_type": "VAEDecode",
            "inputs": {
                "samples": ["5", 0],
                "vae": ["1", 2]
            }
        },
        "7": {
            "class_type": "SaveImage",
            "inputs": {
                "images": ["6", 0],
                "filename_prefix": "ComfyApi"
            }
        }
    }
    return wf

# ----------------------------
# Submit workflow and wait for generated files
# ----------------------------
def submit_workflow_to_comfy(workflow_dict, wait_for_files=True, timeout=CONFIG.COMFY_REQUEST_TIMEOUT):
    """
    Submits the workflow via /prompt and, if wait_for_files=True, polls /history/<prompt_id>
    until outputs contain images. When images are found, the function fetches each image via
    /view and saves them into CONFIG.OUTPUTS_DIR with filename: <prompt_id>_<original_filename>.
    The returned dict uses 'files' as a list of basenames (not absolute paths).
    """
    try:
        resp = comfy_post("/prompt", json_payload={"prompt": workflow_dict}, timeout=timeout)
        j = resp.json()
    except Exception as e:
        return {"error": f"Submission failed: {e}", "probe": probe_comfy()}

    prompt_id = j.get("prompt_id")
    if not prompt_id:
        return {"error": "No prompt_id returned (Comfy rejected workflow)", "meta": j}

    if not wait_for_files:
        return {"task_id": prompt_id, "status": "queued"}

    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            h_resp = comfy_get(f"/history/{prompt_id}")
            history = h_resp.json()
            if prompt_id in history:
                task_data = history[prompt_id]
                files = []
                outputs = task_data.get("outputs", {})

                # Check for error status inside history
                status_obj = task_data.get("status", {})
                if status_obj.get("status_str") == "error":
                    return {"error": "ComfyUI Execution Error", "meta": task_data}

                # Iterate outputs and download any images
                for node_id, node_out in outputs.items():
                    if "images" in node_out:
                        for img in node_out["images"]:
                            fname = img.get("filename")
                            if fname:
                                try:
                                    r_img = comfy_get("/view", params={"filename": fname, "subfolder": img.get("subfolder",""), "type": img.get("type","output")})
                                    local_dir = Path(CONFIG.OUTPUTS_DIR)
                                    local_dir.mkdir(parents=True, exist_ok=True)
                                    local_path = local_dir / f"{prompt_id}_{fname}"
                                    local_path.write_bytes(r_img.content)
                                    files.append(local_path.name)  # store basename for portability
                                except Exception as e:
                                    # non-fatal: continue trying other outputs
                                    print(f"[submit_workflow] failed fetching {fname}: {e}")

                return {"files": files, "meta": task_data}
        except Exception:
            # ignore transient errors and continue polling
            pass
        time.sleep(1.0)

    return {"error": "Timed out waiting for generation", "meta": j}

In [ ]:
# -------------------- 
# Cell 7 (replace entire cell) --------------------
# FastAPI façade: generate, generate-async, status, and view endpoints.
# --------------------
from fastapi import FastAPI, Form, BackgroundTasks, HTTPException
from fastapi.responses import StreamingResponse, JSONResponse
import uuid
import json
from pathlib import Path

app = FastAPI(title="ComfyUI headless t2i façade")

# ---------- Probe endpoint ----------
@app.get("/probe_comfy")
def probe():
    return probe_comfy()

# ---------- Synchronous generate endpoint (returns image stream) ----------
@app.post("/generate")
def generate(prompt: str = Form(...),
             model_manifest: str = Form(None),
             height: int = Form(768), width: int = Form(768),
             num_inference_steps: int = Form(28), guidance_scale: float = Form(7.5),
             seed: int = Form(None)):

    manifest_source = model_manifest or CONFIG.MANIFEST_PATH
    try:
        manifest = read_manifest(manifest_source)
    except Exception as e:
        return JSONResponse({"error": f"manifest read failed: {e}"}, status_code=400)

    try:
        comfy_model_ref = ensure_model_present_for_comfy(manifest)
    except Exception as e:
        return JSONResponse({"error": f"model ensure failed: {e}"}, status_code=500)

    wf = build_minimal_t2i_workflow(prompt, comfy_model_ref, height, width, num_inference_steps, guidance_scale, seed)
    resp = submit_workflow_to_comfy(wf, wait_for_files=True)
    if "error" in resp:
        return JSONResponse({"error": resp}, status_code=500)

    # If files were produced, stream the first file (matching diffusers behavior)
    if resp.get("files"):
        first = resp["files"][0]
        first_path = Path(CONFIG.OUTPUTS_DIR) / first
        if first_path.exists():
            suffix = first_path.suffix.lower()
            ctype = "application/octet-stream"
            if suffix in CONFIG.IMAGE_EXTS:
                ctype = "image/png" if suffix == ".png" else "image/jpeg"
            return StreamingResponse(open(first_path, "rb"), media_type=ctype)
    # otherwise return the metadata
    return JSONResponse(resp)

# ---------- Async generate endpoint (enqueue a background job) ----------
@app.post("/generate-async")
def generate_async(background_tasks: BackgroundTasks,
                   prompt: str = Form(...),
                   model_manifest: str = Form(None),
                   height: int = Form(768), width: int = Form(768),
                   num_inference_steps: int = Form(28), guidance_scale: float = Form(7.5),
                   seed: int = Form(None)):

    manifest_source = model_manifest or CONFIG.MANIFEST_PATH
    try:
        manifest = read_manifest(manifest_source)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"manifest read failed: {e}")

    try:
        comfy_model_ref = ensure_model_present_for_comfy(manifest)
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"model ensure failed: {e}")

    wf = build_minimal_t2i_workflow(prompt, comfy_model_ref, height, width, num_inference_steps, guidance_scale, seed)
    task_id = uuid.uuid4().hex

    def bg_run():
        r = submit_workflow_to_comfy(wf, wait_for_files=True)
        status_file = Path(CONFIG.OUTPUTS_DIR) / f"task_{task_id}.json"
        status_file.parent.mkdir(parents=True, exist_ok=True)
        status_file.write_text(json.dumps(r))

    background_tasks.add_task(bg_run)
    return {"task_id": task_id, "status": "queued"}

# ---------- Status endpoint (polled by your bash client) ----------
@app.get("/status/{task_id}")
def status(task_id: str):
    """
    Polling endpoint:
      - If task not yet present -> JSON {"status":"pending"}
      - If task completed and has files -> RETURNS RAW IMAGE (first file) with image/* content-type
      - Otherwise returns the saved status JSON
    """
    status_file = Path(CONFIG.OUTPUTS_DIR) / f"task_{task_id}.json"
    if not status_file.exists():
        return {"status": "pending"}

    try:
        data = json.loads(status_file.read_text())
    except Exception as e:
        return {"error": f"could not read status file: {e}"}

    files = data.get("files") if isinstance(data, dict) else None
    if files and len(files) > 0:
        candidate = Path(CONFIG.OUTPUTS_DIR) / files[0]
        # fallback: maybe stored was absolute path
        if not candidate.exists():
            candidate_alt = Path(files[0])
            if candidate_alt.exists():
                candidate = candidate_alt

        if candidate.exists():
            suffix = candidate.suffix.lower()
            if suffix == ".png":
                ctype = "image/png"
            elif suffix in [".jpg", ".jpeg"]:
                ctype = "image/jpeg"
            else:
                ctype = "application/octet-stream"
            return StreamingResponse(open(candidate, "rb"), media_type=ctype)

    # No image yet, return JSON status so caller can keep polling
    return data

# ---------- View endpoint (explicit file retrieval) ----------
@app.get("/view")
def view(filename: str):
    p = Path(CONFIG.OUTPUTS_DIR) / filename
    if not p.exists():
        raise HTTPException(status_code=404, detail="file not found")
    suffix = p.suffix.lower()
    ctype = "application/octet-stream"
    if suffix in CONFIG.IMAGE_EXTS:
        ctype = "image/png" if suffix == ".png" else "image/jpeg"
    return StreamingResponse(open(p, "rb"), media_type=ctype)

In [ ]:
# ------------------------------------------------------------------------------
# Cell 8: System Start & Manifest Setup
# ------------------------------------------------------------------------------
print("🚀 Initializing...")

# 1. Create Default Manifest
default_manifest = {
    "name": "sd_xl_base_1.0.safetensors",  # EXACT FILENAME IN CHECKPOINTS
    "source": "local",
    "path": "/kaggle/working/ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors"
}

with open(CONFIG.MANIFEST_PATH, 'w') as f:
    json.dump(default_manifest, f, indent=2)
print(f"✅ Default manifest created at {CONFIG.MANIFEST_PATH}")

# 2. Start ComfyUI
proc = start_comfyui_server()

# 3. Start Zrok
zrok_token = load_zrok_token()
if zrok_token:
    install_and_enable_zrok(zrok_token)
    _, url = start_zrok_share(CONFIG.FACADE_PORT)
    print(f"\n🌍 PUBLIC FACADE URL: {url}\n")

# 4. Start Facade Server
t = threading.Thread(target=uvicorn.run, args=(app,), kwargs={"host":"0.0.0.0", "port":CONFIG.FACADE_PORT}, daemon=True)
t.start()
print(f"✅ Facade running locally at http://0.0.0.0:{CONFIG.FACADE_PORT}")

# 5. Keep Alive
print("🟢 System Ready. Keep-alive active.")
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    if proc: proc.terminate()